In [2]:
# Cell 1: Imports & Setup
import sys
import os
import importlib

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

import models.attention_knn
importlib.reload(models.attention_knn)
from models.attention_knn import AttentionKNN

import core.train_eval
importlib.reload(core.train_eval)
from core.train_eval import train, evaluate

from config import Config

os.makedirs("outputs/logs", exist_ok=True)
os.makedirs("outputs/models", exist_ok=True)
os.makedirs("outputs/plots", exist_ok=True)

print("Setup complete.")


ModuleNotFoundError: No module named 'torch'

In [2]:
# Cell 2: Helper - Run full experiment
def run_experiment(X_train, X_test, y_train, y_test, num_classes,
                   experiment_name="experiment", epochs=None, lr=None, K=None):

    epochs = epochs or Config.epochs
    lr     = lr or Config.lr
    K      = K or Config.K

    print(f"\n{'='*60}")
    print(f"  Experiment: {experiment_name}")
    print(f"  Features: {X_train.shape[1]}, Classes: {num_classes}")
    print(f"  Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")
    print(f"  K={K}, lr={lr}, epochs={epochs}")
    print(f"{'='*60}\n")

    model = AttentionKNN(input_dim=X_train.shape[1], num_classes=num_classes)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    acc_list = []
    loss_list = []

    for epoch in range(epochs):
        loss = train(model, X_train, y_train, optimizer, loss_fn, K)
        acc = evaluate(model, X_test, y_test, X_train, y_train, K)
        loss_list.append(loss)
        acc_list.append(acc)
        if epoch % 5 == 0 or epoch == epochs - 1:
            print(f"  Epoch {epoch:3d}: Loss={loss:.4f}, Acc={acc:.4f}")

    log_path = f"outputs/logs/{experiment_name}_log.txt"
    with open(log_path, "w") as f:
        for epoch in range(epochs):
            f.write(f"Epoch {epoch}: Loss={loss_list[epoch]:.4f}, Acc={acc_list[epoch]:.4f}\n")
    print(f"\n  Log saved: {log_path}")

    model_path = f"outputs/models/{experiment_name}_attn_knn.pt"
    torch.save(model.state_dict(), model_path)
    print(f"  Model saved: {model_path}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(range(epochs), acc_list, color="blue")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Accuracy")
    ax1.set_title(f"{experiment_name} — Accuracy")
    ax1.grid(True, alpha=0.3)
    ax2.plot(range(epochs), loss_list, color="red")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Loss")
    ax2.set_title(f"{experiment_name} — Loss")
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plot_path = f"outputs/plots/{experiment_name}_results.png"
    plt.savefig(plot_path, dpi=150)
    plt.show()
    print(f"  Plot saved: {plot_path}")
    print(f"\n  Final Accuracy: {acc_list[-1]:.4f}")
    print(f"  Best Accuracy:  {max(acc_list):.4f} (Epoch {acc_list.index(max(acc_list))})")

    return acc_list, loss_list


In [ ]:
# Cell 3: Experiment 1 - Image Classification (CIFAR-10)
from data.image_classification import load_data as load_image_data

X_train_img, X_test_img, y_train_img, y_test_img, num_classes_img = load_image_data()

img_acc, img_loss = run_experiment(
    X_train_img, X_test_img, y_train_img, y_test_img,
    num_classes=num_classes_img,
    experiment_name="cifar10_image_classification"
)


 10%|█         | 17.6M/170M [00:27<02:50, 895kB/s] 

In [ ]:
# Cell 4: Experiment 2 - Sentiment Analysis
from data.sentiment_analysis import load_data as load_sentiment_data

X_train_txt, X_test_txt, y_train_txt, y_test_txt, num_classes_txt = load_sentiment_data()

txt_acc, txt_loss = run_experiment(
    X_train_txt, X_test_txt, y_train_txt, y_test_txt,
    num_classes=num_classes_txt,
    experiment_name="sentiment_analysis",
    K=10,
    epochs=50,
    lr=0.0003
)


In [ ]:
# Cell 5: Comparison Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(len(img_acc)), img_acc, label="CIFAR-10 (Image)", color="blue")
ax1.plot(range(len(txt_acc)), txt_acc, label="Sentiment (Text)", color="green")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.set_title("Attn-KNN — Accuracy Comparison")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(range(len(img_loss)), img_loss, label="CIFAR-10 (Image)", color="blue")
ax2.plot(range(len(txt_loss)), txt_loss, label="Sentiment (Text)", color="green")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.set_title("Attn-KNN — Loss Comparison")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/plots/comparison_results.png", dpi=150)
plt.show()
print("Comparison plot saved.")


In [ ]:
# Cell 6: Summary Table
print(f"\n{'='*55}")
print(f"{'Experiment':<35} {'Best Acc':>8} {'Final Acc':>10}")
print(f"{'='*55}")
print(f"{'CIFAR-10 Image Classification':<35} {max(img_acc):>8.4f} {img_acc[-1]:>10.4f}")
print(f"{'Sentiment Analysis':<35} {max(txt_acc):>8.4f} {txt_acc[-1]:>10.4f}")
print(f"{'='*55}")
